In [2]:
import pandas as pd
import numpy as np

# Основная таблица с сотрудниками
df = pd.DataFrame({
    'emp_id': [101, 102, 103, 104, 105, 106, 107, 108],
    'name': ['Анна', 'Борис', 'Виктор', 'Галина', 'Дмитрий', 'Елена', 'Жанна', 'Захар'],
    'dept': ['IT', 'Sales', 'IT', 'HR', 'Sales', 'IT', 'HR', 'Sales'],
    'salary': [60000, 75000, 80000, 70000, 65000, 82000, 72000, 68000],
    'age': [25, 30, 35, 42, 28, 31, 29, 33],
    'hours': [120, 80, 150, 90, 60, 130, 110, 95],
    'city': ['Москва', 'СПб', 'Москва', 'Казань', 'СПб', 'Москва', 'Казань', 'СПб']
})

# Вторая таблица для merge
bonuses = pd.DataFrame({
    'dept': ['IT', 'Sales', 'HR', 'Marketing'],
    'bonus_percent': [0.2, 0.15, 0.1, 0.25]
})

# Третья таблица для merge
projects = pd.DataFrame({
    'emp_id': [101, 102, 103, 104, 105, 106, 107, 101, 103],
    'project': ['P1', 'P2', 'P3', 'P1', 'P2', 'P3', 'P4', 'P4', 'P1'],
    'project_hours': [120, 80, 150, 90, 60, 130, 110, 70, 140]
})

In [6]:
df['new'] = df.groupby('dept')['salary'].transform('rank', ascending=False)

df[df['new'] == 1]

,emp_id,name,dept,salary,age,hours,city,new
1,102,Борис,Sales,75000,30,80,СПб,1.0
5,106,Елена,IT,82000,31,130,Москва,1.0
6,107,Жанна,HR,72000,29,110,Казань,1.0


In [3]:
df.groupby('dept').apply(lambda x: x.nlargest(1, 'salary')).reset_index(drop=True)

,emp_id,name,salary,age,hours,city,new
0,107,Жанна,72000,29,110,Казань,2.0
1,106,Елена,82000,31,130,Москва,3.0
2,102,Борис,75000,30,80,СПб,3.0


In [14]:
df.drop_duplicates('name')

,emp_id,name,dept,salary,age,hours,city,new
0,101,Анна,IT,60000,25,120,Москва,3.0
1,102,Борис,Sales,75000,30,80,СПб,1.0
2,103,Виктор,IT,80000,35,150,Москва,2.0
3,104,Галина,HR,70000,42,90,Казань,2.0
4,105,Дмитрий,Sales,65000,28,60,СПб,3.0
5,106,Елена,IT,82000,31,130,Москва,1.0
6,107,Жанна,HR,72000,29,110,Казань,1.0
7,108,Захар,Sales,68000,33,95,СПб,2.0


In [30]:
df = pd.DataFrame({'column': [1, 1, 1, 2, 2, 1]})

df['shifted'] = df['column'].shift()
df['shifted 1'] = df['column'].shift(1)
df['shifted 2'] = df['column'].shift(2)
df['mask'] = df['column'] != df['column'].shift()
df['streak_id'] = df['mask'].cumsum()
df

,column,shifted,shifted 1,shifted 2,mask,streak_id
0,1,NaN,NaN,NaN,True,1
1,1,1.0,1.0,NaN,False,1
2,1,1.0,1.0,1.0,False,1
3,2,1.0,1.0,1.0,True,2
4,2,2.0,2.0,1.0,False,2
5,1,2.0,2.0,2.0,True,3


In [41]:
df['column'][df['column'].duplicated()].unique()

array([1, 2])

In [17]:
test = pd.DataFrame({'value': [1, 1, 1, 0, 0, 1]})
test['id'] = pd.Series([1, 2, 3, 4, 5, 6])

test['group'] = (test['value'] != test['value'].shift())

test['id_shifted'] = test['id'].shift()
test['ids'] = (test['id'] - 1 != test['id'].shift()).sum()
test

,value,id,group,id_shifted,ids
0,1,1,True,NaN,1
1,1,2,False,1.0,1
2,1,3,False,2.0,1
3,0,4,True,3.0,1
4,0,5,False,4.0,1
5,1,6,True,5.0,1


In [21]:
stadium = {
    'id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17],
    'visit_date': ['2017-5-1', '2017-5-2', '2017-5-3', '2017-5-4', '2017-5-5', 
                   '2017-5-6', '2017-5-7', '2017-5-8', '2017-5-9', '2017-5-10',
                   '2017-5-11', '2017-5-12', '2017-5-13', '2017-5-14', '2017-5-15',
                   '2017-5-16', '2017-5-17'],
    'people': [150, 150, 150, 150, 50, 50, 150, 150, 150, 150, 150, 50, 50, 150, 150, 150, 150]
}

stadium = pd.DataFrame(stadium)



In [38]:
df = stadium.sort_values(by='id')

df['morethan100'] = df['people'] >= 100

df['group'] = (df['morethan100'] != df['morethan100'].shift()).cumsum()

df = df[df['morethan100']]

df = df.groupby('group').filter(lambda x: len(x) >= 3 and (x['id'].diff().fillna(1) == 1).all())

df['visit_date'] = pd.to_datetime(df['visit_date'])

df = df.sort_values('visit_date')

df = df[['id', 'visit_date', 'people']]

# df = df.groupby('group').filter(lambda x: len(x) >= 3 and df['morethan100'].all() and (x['id'].diff().fillna(1) == 1).all())

df

,id,visit_date,people
0,1,2017-05-01,150
1,2,2017-05-02,150
2,3,2017-05-03,150
3,4,2017-05-04,150
6,7,2017-05-07,150
7,8,2017-05-08,150
8,9,2017-05-09,150
9,10,2017-05-10,150
10,11,2017-05-11,150
13,14,2017-05-14,150


In [ ]:
df = stadium.sort_values(by='id')

df['morethan100'] = df['people'] >= 100

df['group'] = (df['morethan100'] != df['morethan100'].shift()).cumsum()

df = df.groupby('group').filter(lambda x: len(x) >= 3 and df['morethan100'].all() and (x['id'].diff().fillna(1) == 1).all())

df = df.sort_values('visit_date')

# df = df[['id', 'visit_date', 'people']]

df


,id,visit_date,people,morethan100,group
0,1,2017-5-1,150,True,1
9,10,2017-5-10,150,True,3
10,11,2017-5-11,150,True,3
13,14,2017-5-14,150,True,5
14,15,2017-5-15,150,True,5
15,16,2017-5-16,150,True,5
16,17,2017-5-17,150,True,5
1,2,2017-5-2,150,True,1
2,3,2017-5-3,150,True,1
3,4,2017-5-4,150,True,1
